In [ ]:
%load_ext autoreload
%autoreload 2


# Flight Delay ML Platform — Scheduled Congestion Features

## 1. Research Objective

Investigate whether scheduled airport traffic provides
incremental predictive value beyond individual flight
schedule information.

## Feature Source

Scheduled traffic statistics are constructed from raw
BTS monthly files without filtering flights based on
their observed operational outcomes.

The prediction cohort remains restricted to completed,
non-diverted flights with valid target labels.

## Initial Features

- Origin scheduled departures within the same clock hour
- Airline-origin scheduled departures within the same
  clock hour

## Leakage Control

- Do not use cancellation, diversion, or delay outcomes
  when constructing congestion statistics.
- Use scheduled departure times only.
- Preserve the established prediction cohort.
- Restrict development to January 2023–August 2024.
- Keep calibration, threshold-selection, and final test
  datasets excluded.

## Limitation

BTS records provide a retrospective schedule proxy,
not a verified historical T-24h schedule snapshot.

Scheduled traffic counts represent reporting-carrier
activity rather than complete airport traffic or
actual airport congestion.


In [ ]:
import json
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from flight_delay.data.manifest import sha256_file
from flight_delay.data.storage import load_parquet

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

assert (ROOT / "pyproject.toml").is_file()

RAW_DIR = ROOT / "data/raw"
PROCESSED_DIR = ROOT / "data/processed"
MANIFEST_DIR = ROOT / "data/interim/manifests"

ORIGIN_AIRPORTS = [
    "ATL", "DFW", "DEN", "ORD", "LAX", "SFO"
]

RAW_SCHEDULE_COLUMNS = [
    "FlightDate",
    "Origin",
    "Reporting_Airline",
    "CRSDepTime",
]

print("Project root:", ROOT)
print("Target origin airports:", ORIGIN_AIRPORTS)

In [ ]:
def load_raw_departure_schedule(month):

    if month not in set(
        pd.period_range(
            "2023-01", "2024-08", freq="M"
        ).astype(str)
    ):
        raise ValueError(
            f"Month outside development scope: {month}"
        )

    with (MANIFEST_DIR / f"{month}.json").open(
        encoding="utf-8"
    ) as f:
        manifest = json.load(f)

    path = RAW_DIR / manifest["source"]["filename"]

    assert path.is_file()
    assert sha256_file(path) == manifest["source"]["sha256"]

    with zipfile.ZipFile(path) as archive:

        csv_members = [
            name for name in archive.namelist()
            if name.lower().endswith(".csv")
        ]

        assert len(csv_members) == 1

        with archive.open(csv_members[0]) as f:

            schedule = pd.read_csv(
                f,
                usecols=RAW_SCHEDULE_COLUMNS,
                dtype={
                    "Origin": "string",
                    "Reporting_Airline": "string",
                },
                low_memory=False,
            )

    return schedule


start = time.perf_counter()

jan_schedule = load_raw_departure_schedule(
    "2023-01"
)

print("=== January Raw Schedule ===")
print("Total raw records:", len(jan_schedule))
print("Loaded columns:", jan_schedule.columns.tolist())

print(
    "Loading seconds:",
    round(time.perf_counter() - start, 2),
)

display(jan_schedule.head())

print("\nMissing scheduled inputs:")
display(jan_schedule.isna().sum())

In [ ]:
def prepare_departure_schedule(schedule, month):

    result = schedule.loc[
        schedule["Origin"].isin(ORIGIN_AIRPORTS)
    ].copy()

    dates = pd.to_datetime(
        result["FlightDate"],
        errors="coerce",
    )

    valid_dates = (
        dates.notna()
        & dates.dt.to_period("M").eq(pd.Period(month))
    )

    numeric = pd.to_numeric(
        result["CRSDepTime"],
        errors="coerce",
    )

    unresolved_2400 = numeric.eq(2400).fillna(False)

    valid_clock = (
        numeric.notna()
        & numeric.between(0, 2359)
        & numeric.mod(100).between(0, 59)
        & numeric.mod(1).eq(0)
    ).fillna(False)

    valid_airline = (
        result["Reporting_Airline"].notna()
        & result["Reporting_Airline"]
        .str.strip().ne("")
    ).fillna(False)

    valid = (
        valid_dates
        & valid_clock
        & valid_airline
    )

    audit = {
        "Month": month,
        "ScopedRecords": len(result),
        "InvalidDates": int((~valid_dates).sum()),
        "MissingOrInvalidTimes": int(
            (~valid_clock & ~unresolved_2400).sum()
        ),
        "Unresolved2400": int(unresolved_2400.sum()),
        "MissingAirline": int((~valid_airline).sum()),
        "IncludedRecords": int(valid.sum()),
        "ExcludedRecords": int((~valid).sum()),
    }

    result = result.loc[valid].copy()

    result["FlightDate"] = (
        dates.loc[valid].dt.normalize()
    )

    result["Departure_Hour"] = (
        numeric.loc[valid].astype("int64") // 100
    ).astype("int8")

    return result, audit


jan_departure_schedule, jan_schedule_audit = (
    prepare_departure_schedule(
        jan_schedule,
        "2023-01",
    )
)

print("=== Scheduled Departure Quality Audit ===")

display(pd.DataFrame([jan_schedule_audit]))

print(
    "Valid scheduled records:",
    len(jan_departure_schedule),
)

display(jan_departure_schedule.head())


## 2. Scheduled Departure Density

Scheduled departure density is calculated using fixed
local clock-hour buckets.

For each flight, origin traffic represents the number
of eligible scheduled departure records at the same
origin, on the same date, during the same clock hour.

Airline-origin traffic further restricts this count
to the corresponding reporting airline.

The counts include the target flight itself.

These features measure scheduled traffic density,
not directly observed airport congestion.


In [ ]:
ORIGIN_TRAFFIC_KEYS = [
    "FlightDate",
    "Origin",
    "Departure_Hour",
]

AIRLINE_TRAFFIC_KEYS = [
    "FlightDate",
    "Origin",
    "Departure_Hour",
    "Reporting_Airline",
]

DEPARTURE_TRAFFIC_FEATURES = [
    "origin_scheduled_departures_1h",
    "airline_origin_scheduled_departures_1h",
]


def aggregate_departure_traffic(schedule):

    origin_traffic = (
        schedule
        .groupby(
            ORIGIN_TRAFFIC_KEYS,
            observed=True,
            sort=True,
        )
        .size()
        .reset_index(
            name=DEPARTURE_TRAFFIC_FEATURES[0]
        )
    )

    airline_traffic = (
        schedule
        .groupby(
            AIRLINE_TRAFFIC_KEYS,
            observed=True,
            sort=True,
        )
        .size()
        .reset_index(
            name=DEPARTURE_TRAFFIC_FEATURES[1]
        )
    )

    assert origin_traffic[
        DEPARTURE_TRAFFIC_FEATURES[0]
    ].sum() == len(schedule)

    assert airline_traffic[
        DEPARTURE_TRAFFIC_FEATURES[1]
    ].sum() == len(schedule)

    assert not origin_traffic.duplicated(
        ORIGIN_TRAFFIC_KEYS
    ).any()

    assert not airline_traffic.duplicated(
        AIRLINE_TRAFFIC_KEYS
    ).any()

    return origin_traffic, airline_traffic


jan_origin_traffic, jan_airline_traffic = (
    aggregate_departure_traffic(
        jan_departure_schedule
    )
)

print("=== January Airport Traffic ===")

print(
    "Origin-hour groups:",
    len(jan_origin_traffic),
)

print(
    "Airline-origin-hour groups:",
    len(jan_airline_traffic),
)

display(jan_origin_traffic.head(10))
display(jan_airline_traffic.head(10))

print("\nTraffic aggregation passed!")

In [ ]:
reaggregated = (
    jan_airline_traffic
    .groupby(ORIGIN_TRAFFIC_KEYS)[
        "airline_origin_scheduled_departures_1h"
    ]
    .sum()
    .sort_index()
)

expected = (
    jan_origin_traffic
    .set_index(ORIGIN_TRAFFIC_KEYS)[
        "origin_scheduled_departures_1h"
    ]
    .sort_index()
)

pd.testing.assert_series_equal(
    reaggregated,
    expected,
    check_names=False,
    check_dtype=False,
)

print("Airline-origin traffic reconciliation passed!")

In [ ]:
from flight_delay.features.advanced_scheduled import (
    hhmm_to_hour,
)

with (MANIFEST_DIR / "2023-01.json").open(
    encoding="utf-8"
) as f:
    jan_manifest = json.load(f)

jan_path = (
    PROCESSED_DIR
    / jan_manifest["processed"]["filename"]
)

assert sha256_file(jan_path) == (
    jan_manifest["processed"]["sha256"]
)

jan_target = load_parquet(jan_path)

assert len(jan_target) == 112_548

# January was previously found to have no unresolved
# departure timestamps.
assert not jan_target["CRSDepTime"].eq(2400).any()

target_keys = jan_target[
    ["FlightDate", "Origin", "Reporting_Airline"]
].copy()

target_keys["FlightDate"] = pd.to_datetime(
    target_keys["FlightDate"]
).dt.normalize()

for column in ["Origin", "Reporting_Airline"]:
    target_keys[column] = (
        target_keys[column].astype("string")
    )

target_keys["Departure_Hour"] = hhmm_to_hour(
    jan_target["CRSDepTime"]
)

target_keys["_row_id"] = np.arange(
    len(target_keys)
)

jan_traffic_joined = (
    target_keys
    .merge(
        jan_origin_traffic,
        on=ORIGIN_TRAFFIC_KEYS,
        how="left",
        validate="many_to_one",
    )
    .merge(
        jan_airline_traffic,
        on=AIRLINE_TRAFFIC_KEYS,
        how="left",
        validate="many_to_one",
    )
    .sort_values("_row_id")
    .reset_index(drop=True)
)

assert len(jan_traffic_joined) == len(jan_target)

np.testing.assert_array_equal(
    jan_traffic_joined["_row_id"].to_numpy(),
    np.arange(len(jan_target)),
)

print("=== January Congestion Feature Pilot ===")

display(jan_traffic_joined.head(10))

print("\nMissing features:")

display(
    jan_traffic_joined[
        DEPARTURE_TRAFFIC_FEATURES
    ].isna().sum()
)

print("\nTraffic feature statistics:")

display(
    jan_traffic_joined[
        DEPARTURE_TRAFFIC_FEATURES
    ].describe()
)

In [ ]:
traffic_features = jan_traffic_joined[
    DEPARTURE_TRAFFIC_FEATURES
]

assert traffic_features.notna().all().all()

assert traffic_features.ge(1).all().all()

assert (
    traffic_features[
        "airline_origin_scheduled_departures_1h"
    ]
    <= traffic_features[
        "origin_scheduled_departures_1h"
    ]
).all()

# Completed-flight counts are used only for auditing,
# not for constructing the model features.
completed_counts = (
    target_keys
    .groupby(ORIGIN_TRAFFIC_KEYS)
    .size()
    .reset_index(name="completed_count")
)

comparison = completed_counts.merge(
    jan_origin_traffic,
    on=ORIGIN_TRAFFIC_KEYS,
    how="left",
    validate="one_to_one",
)

assert comparison[
    "origin_scheduled_departures_1h"
].notna().all()

assert (
    comparison["origin_scheduled_departures_1h"]
    >= comparison["completed_count"]
).all()

print("=== Congestion Feature Validation ===")
print("Record alignment: Passed")
print("Feature coverage: Passed")
print("Positive traffic counts: Passed")
print("Airline-origin consistency: Passed")
print("Raw vs. completed reconciliation: Passed")

print("\nJanuary congestion feature pilot passed!")

In [ ]:
import matplotlib.pyplot as plt

average_hourly_traffic = (
    jan_origin_traffic
    .groupby(["Origin", "Departure_Hour"])[
        "origin_scheduled_departures_1h"
    ]
    .mean()
    .unstack("Origin")
)

display(average_hourly_traffic.round(2))

ax = average_hourly_traffic.plot(
    figsize=(12, 5),
    marker="o",
    markersize=3,
)

ax.set_title(
    "January 2023: Average Scheduled Departures "
    "by Airport and Hour"
)

ax.set_xlabel("Local Departure Hour")
ax.set_ylabel("Mean Departures per Active Hour")
ax.set_xticks(range(24))
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 3. Modular Scheduled Traffic Feature Pipeline

The scheduled departure traffic pipeline is implemented
as a reusable Python module.

Regression verification compares the modular
implementation against the previously validated
January 2023 notebook implementation.

The feature definitions and prediction cohort remain
unchanged.


In [ ]:
import flight_delay.features.scheduled_congestion as sc

new_schedule, new_audit = (
    sc.prepare_departure_schedule(
        jan_schedule,
        "2023-01",
    )
)

new_origin_traffic, new_airline_traffic = (
    sc.aggregate_departure_traffic(
        new_schedule
    )
)

new_jan_features = (
    sc.build_departure_traffic_features(
        flights=jan_target,
        origin_traffic=new_origin_traffic,
        airline_traffic=new_airline_traffic,
        month="2023-01",
    )
)

pd.testing.assert_frame_equal(
    new_jan_features.reset_index(drop=True),
    traffic_features.reset_index(drop=True),
    check_dtype=False,
)

assert new_audit == jan_schedule_audit

assert new_jan_features.shape == (112_548, 2)

print("January schedule records:", len(new_schedule))
print("Origin-hour groups:", len(new_origin_traffic))
print("Airline-hour groups:", len(new_airline_traffic))
print("Feature shape:", new_jan_features.shape)

print("\nJanuary regression verification passed!")


## 4. Full Scheduled Departure Traffic Generation

Generate two scheduled traffic features for all
training and development validation flights.

Source files are verified against their monthly
manifest checksums.

Monthly traffic aggregation is performed independently
without filtering on observed flight outcomes.

The generated feature records preserve the original
flight candidate keys for subsequent model integration.

Unresolved schedule inputs and unmatched records are
reported rather than silently imputed.


In [ ]:
IDENTITY_COLUMNS = [
    "FlightDate",
    "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
]

development_months = pd.period_range(
    "2023-01",
    "2024-08",
    freq="M",
).astype(str)

congestion_features_by_month = {}
congestion_audit_records = []

start = time.perf_counter()

for month in development_months:

    # Read raw schedules and verify source checksum.
    raw_schedule = load_raw_departure_schedule(
        month
    )

    prepared, audit = sc.prepare_departure_schedule(
        raw_schedule,
        month,
    )

    # Fail closed if any scheduled records are unresolved.
    if audit["ExcludedRecords"] != 0:
        print(audit)
        raise ValueError(
            f"Unresolved scheduled records: {month}"
        )

    origin_traffic, airline_traffic = (
        sc.aggregate_departure_traffic(prepared)
    )

    # Independently verify prediction cohort.
    with (MANIFEST_DIR / f"{month}.json").open(
        encoding="utf-8"
    ) as f:
        manifest = json.load(f)

    path = (
        PROCESSED_DIR
        / manifest["processed"]["filename"]
    )

    assert sha256_file(path) == (
        manifest["processed"]["sha256"]
    )

    target = load_parquet(path)

    assert len(target) == (
        manifest["processed"]["row_count"]
    )

    features = sc.build_departure_traffic_features(
        flights=target,
        origin_traffic=origin_traffic,
        airline_traffic=airline_traffic,
        month=month,
    )

    assert features.index.equals(target.index)

    assert features.shape == (len(target), 2)
    assert not features.isna().any().any()

    saved_frame = pd.concat(
        [
            target[IDENTITY_COLUMNS],
            features,
        ],
        axis=1,
    )

    assert not saved_frame[
        IDENTITY_COLUMNS
    ].isna().any().any()

    assert not saved_frame.duplicated(
        IDENTITY_COLUMNS
    ).any()

    congestion_features_by_month[month] = saved_frame

    congestion_audit_records.append({
        **audit,
        "RawRecords": len(raw_schedule),
        "TargetRecords": len(target),
        "OriginHourGroups": len(origin_traffic),
        "AirlineHourGroups": len(airline_traffic),
        "CompleteFeatures": int(
            features.notna().all(axis=1).sum()
        ),
        "OriginTrafficMean": features.iloc[:, 0].mean(),
        "AirlineTrafficMean": features.iloc[:, 1].mean(),
    })

    print(
        f"{month} | "
        f"scheduled={len(prepared):,} | "
        f"targets={len(target):,} | "
        f"coverage={features.notna().all(axis=1).mean():.6%}"
    )

    del raw_schedule, prepared
    del origin_traffic, airline_traffic
    del target, features, saved_frame

congestion_audit = pd.DataFrame(
    congestion_audit_records
)

print(
    "\nGeneration seconds:",
    round(time.perf_counter() - start, 2),
)

display(congestion_audit)

print("\nFull scheduled traffic generation completed!")

In [ ]:
assert len(congestion_audit) == 20

assert set(congestion_features_by_month) == set(
    development_months
)

assert congestion_audit["TargetRecords"].sum() == 2_498_494

assert (
    congestion_audit["CompleteFeatures"].sum()
    == congestion_audit["TargetRecords"].sum()
)

assert congestion_audit["ExcludedRecords"].sum() == 0

assert (
    congestion_audit["IncludedRecords"]
    >= congestion_audit["TargetRecords"]
).all()

for month, frame in congestion_features_by_month.items():

    expected_columns = (
        IDENTITY_COLUMNS
        + list(sc.DEPARTURE_TRAFFIC_FEATURES)
    )

    assert frame.columns.tolist() == expected_columns

    assert not frame.duplicated(
        IDENTITY_COLUMNS
    ).any()

    traffic = frame[
        list(sc.DEPARTURE_TRAFFIC_FEATURES)
    ]

    assert not traffic.isna().any().any()
    assert traffic.ge(1).all().all()

    assert (
        traffic.iloc[:, 1] <= traffic.iloc[:, 0]
    ).all()

total_targets = congestion_audit["TargetRecords"].sum()
total_complete = congestion_audit["CompleteFeatures"].sum()

print("=== Full Congestion Feature Summary ===")

print("Months:", len(congestion_audit))

print(
    "Total raw records:",
    f"{congestion_audit['RawRecords'].sum():,}"
)

print(
    "In-scope scheduled flights:",
    f"{congestion_audit['IncludedRecords'].sum():,}"
)

print("Target flights:", f"{total_targets:,}")

print("Complete features:", f"{total_complete:,}")

print(
    "Feature coverage:",
    f"{total_complete / total_targets:.6%}",
)

print("Excluded records:", congestion_audit["ExcludedRecords"].sum())

print("\nFull congestion feature audit passed!")

In [ ]:
ax = congestion_audit.set_index("Month")[
    ["OriginTrafficMean", "AirlineTrafficMean"]
].plot(
    figsize=(12, 5),
    marker="o",
)

ax.axvline(
    x=11.5,
    linestyle="--",
    color="gray",
    label="Train/Validation Boundary",
)

ax.set_title(
    "Monthly Scheduled Departure Traffic"
)

ax.set_ylabel(
    "Mean Scheduled Flights per Target Flight's Hour"
)

ax.set_xlabel("Month")
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()